<a href="https://colab.research.google.com/github/devharsh/cyberquest-camp/blob/main/Day3/Day3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Day 3 Notebook: Cryptography

Run a cell with **Shift + Enter**. Work through the Modules in order.

**Modules in this notebook:**
1. Module 1: Caesar cipher and brute force
2. Module 2: Base64 and XOR
3. Module 3: Hashing and the avalanche effect
4. Module 4: RSA with tiny numbers

The slides tell you when to run each Module. After each Module, go back to the slides.


## Module 1: Caesar cipher and brute force
Shift each letter by a fixed amount. With only 25 shifts, you can brute-force it.


In [ ]:
def caesar(t, s):
    o = ""
    for c in t:
        if c.isalpha():
            b = ord("A") if c.isupper() else ord("a")
            o += chr((ord(c) - b + s) % 26 + b)
        else: o += c
    return o

ct = caesar("Attack at dawn", 3)
print("cipher:", ct)
for s in range(1, 26):
    print(s, caesar(ct, -s))


## Module 2: Base64 and XOR
Base64 is encoding (anyone can reverse). XOR with the same key twice returns the original (a tiny taste of stream ciphers).


In [ ]:
import base64
m = "meet at noon"
enc = base64.b64encode(m.encode()).decode()
print("base64:", enc, "->", base64.b64decode(enc).decode())

def xor(d,k): return bytes(b ^ k for b in d)
c = xor(b"flag{xor}", 42)
print("xor back:", xor(c, 42).decode())


## Module 3: Hashing and the avalanche effect
Change one character and the whole SHA-256 fingerprint changes. Hashes are one-way.


In [ ]:
import hashlib
for w in ["password", "Password", "password!"]:
    print(w, "->", hashlib.sha256(w.encode()).hexdigest()[:20])


## Module 4: RSA with tiny numbers
RSA uses two keys. With tiny primes you can follow the math by hand. p=5, q=11, n=55. Public key (e=3, n=55), private key (d=27, n=55).


In [ ]:
def rsa(m, key):
    e, n = key
    return pow(m, e, n)

pub = (3, 55)
priv = (27, 55)
msg = 7
c = rsa(msg, pub)
print("encrypted:", c)
print("decrypted:", rsa(c, priv))
# decrypting the cipher returns the original 7


## Going further: more classical ciphers

Building on Modules 1 to 4, these go beyond the Caesar cipher.

### Vigenere Cipher

The Vigenere cipher uses a keyword to shift each letter by a different amount.
It resisted cryptanalysis for centuries -- until Kasiski broke it in 1863.


In [ ]:
def vigenere_encrypt(text: str, key: str) -> str:
    """Encrypt text using Vigenere cipher."""
    key = key.upper()
    result = []
    key_idx = 0
    for ch in text:
        if ch.isalpha():
            shift = ord(key[key_idx % len(key)]) - ord('A')
            base  = ord('A') if ch.isupper() else ord('a')
            result.append(chr((ord(ch) - base + shift) % 26 + base))
            key_idx += 1
        else:
            result.append(ch)
    return ''.join(result)

def vigenere_decrypt(text: str, key: str) -> str:
    """Decrypt Vigenere cipher given the key."""
    key = key.upper()
    result = []
    key_idx = 0
    for ch in text:
        if ch.isalpha():
            shift = ord(key[key_idx % len(key)]) - ord('A')
            base  = ord('A') if ch.isupper() else ord('a')
            result.append(chr((ord(ch) - base - shift) % 26 + base))
            key_idx += 1
        else:
            result.append(ch)
    return ''.join(result)

# Demo
plaintext = 'Attack at dawn'
key       = 'LEMON'
encrypted = vigenere_encrypt(plaintext, key)
decrypted = vigenere_decrypt(encrypted, key)
print(f'Plaintext : {plaintext}')
print(f'Key       : {key}')
print(f'Encrypted : {encrypted}')
print(f'Decrypted : {decrypted}')
# Expected: Encrypted: Lxfopv ef rnhr


### Frequency Analysis

English text has predictable letter frequencies (E is most common, Z is least).
Frequency analysis exploits this to break monoalphabetic substitution ciphers.


In [ ]:
from collections import Counter

# English letter frequencies (approximate %)
ENGLISH_FREQ = {
    'E':12.7,'T':9.1,'A':8.2,'O':7.5,'I':7.0,'N':6.7,'S':6.3,'H':6.1,
    'R':6.0,'D':4.3,'L':4.0,'C':2.8,'U':2.8,'M':2.4,'W':2.4,'F':2.2,
    'G':2.0,'Y':2.0,'P':1.9,'B':1.5,'V':1.0,'K':0.8,'J':0.2,'X':0.2,
    'Q':0.1,'Z':0.1
}

def frequency_analysis(text: str) -> list:
    """Count letter frequencies, return sorted list of (letter, count, %)."""
    letters = [ch.upper() for ch in text if ch.isalpha()]
    total   = len(letters)
    counts  = Counter(letters)
    return [(ch, cnt, 100*cnt/total) for ch, cnt in counts.most_common()]

sample = """When in the course of human events it becomes necessary for one people
to dissolve the political bands which have connected them with another"""

print('Letter frequencies in sample text:')
for letter, count, pct in frequency_analysis(sample)[:8]:
    bar = '#' * int(pct)
    print(f'  {letter}: {pct:5.1f}%  {bar}')


### Student Challenge: Frequency Analysis Attack

The ciphertext below was encrypted with a simple monoalphabetic substitution cipher.
Use frequency analysis to find the key and decrypt it.

**Hint:** Find the most common letter in the ciphertext, assume it maps to 'E',
then try different Caesar shifts.


In [ ]:
# ============ STUDENT EXERCISE ============
# The message below was encrypted with a Caesar cipher.
# Use frequency analysis to find the shift WITHOUT brute-forcing.
#
# TODO:
#   1. Run frequency_analysis() on cipher_challenge
#   2. Find the most frequent letter
#   3. Assume it maps to 'E' -- calculate the shift
#   4. Decrypt and verify the message makes sense
#
# Expected output: the plaintext is a famous security quote
# ==========================================

cipher_challenge = (
    'Kyv jkifeyvjk gligpd zj kyv fev pfl tiv'
    'RkV pfl ijj'
    'Znz lj Kyviv zi v efe vj f jkife z'
)

# Simpler, actually solvable challenge:
cipher_challenge = 'Ymj xjhzwnyd tk f xdxyjr nx tsqd fx xywtrj fx nyx bjfpjxy qnsp.'

# TODO: use frequency_analysis() defined above and caesar_decrypt() to solve this
freqs = frequency_analysis(cipher_challenge)
print('Top 5 letters in ciphertext:')
for letter, count, pct in freqs[:5]:
    print(f'  {letter}: {pct:.1f}%')

# TODO: calculate shift and decrypt
# shift = ?
# plaintext = caesar_decrypt(cipher_challenge, shift)
# print('Plaintext:', plaintext)


## Going further: hashing and integrity

Deeper hashing tools used for message authentication and safe password storage.

### HMAC: Keyed Hash for Message Authentication

HMAC adds a **secret key** to the hash so only someone with the key can verify
the message has not been tampered with. Used in JWT tokens, API authentication, etc.


In [ ]:
import hashlib
import hmac as hmac_lib

key     = b'supersecretkey'
message = b'Transfer $100 to Alice'

# Create HMAC-SHA256
mac = hmac_lib.new(key, message, hashlib.sha256).hexdigest()
print(f'Message : {message.decode()}')
print(f'HMAC    : {mac}')

# Tampered message -- HMAC will not match
tampered = b'Transfer $10000 to Alice'
mac_tampered = hmac_lib.new(key, tampered, hashlib.sha256).hexdigest()
print(f'\nTampered: {tampered.decode()}')
print(f'HMAC    : {mac_tampered}')
print(f'MACs match: {mac == mac_tampered}')  # Expected: False

# Proper constant-time comparison (prevents timing attacks)
print('\nSecure verification (constant-time):')
print(f'  Original match : {hmac_lib.compare_digest(mac, mac)}')
print(f'  Tampered match : {hmac_lib.compare_digest(mac, mac_tampered)}')


### Password Hashing with PBKDF2-HMAC and Salt

**Never store passwords as plain hashes!**
Attackers precompute rainbow tables of common password hashes.
Adding a random **salt** makes each hash unique even for identical passwords.
**Key stretching** (many iterations) makes brute force slow.


In [ ]:
import hashlib, os, binascii

def hash_password(password: str) -> str:
    """Hash a password with PBKDF2-HMAC-SHA256 + random salt."""
    salt       = os.urandom(32)           # 256-bit random salt
    iterations = 260_000                  # NIST recommended minimum
    dk = hashlib.pbkdf2_hmac('sha256', password.encode(), salt, iterations)
    # Store as: iterations$salt_hex$hash_hex
    return f'{iterations}${salt.hex()}${dk.hex()}'

def verify_password(password: str, stored_hash: str) -> bool:
    """Verify a password against a stored PBKDF2 hash."""
    iterations, salt_hex, hash_hex = stored_hash.split('$')
    salt = bytes.fromhex(salt_hex)
    dk   = hashlib.pbkdf2_hmac('sha256', password.encode(), salt, int(iterations))
    return hmac_lib.compare_digest(dk.hex(), hash_hex)

# Demo
pwd = 'MyP@ssw0rd!'
stored = hash_password(pwd)
print(f'Stored hash: {stored[:60]}...')
print(f'Verify correct  : {verify_password(pwd, stored)}')
print(f'Verify incorrect: {verify_password("wrongpassword", stored)}')
# Even same password = different stored hash due to random salt:
stored2 = hash_password(pwd)
print(f'Same password, same hash: {stored == stored2}')  # Expected: False


### Student Challenge: Hash Identification

**From the Integrity.ipynb exercises:**

Given the hash: `C4F3ACA26A244839355883DDC951083C1056FFBC`

1. How many hex characters is it? What algorithm does that suggest?
2. Which of {`Stevens`, `stevens`, `STEVENS`} produced this hash?


In [ ]:
# ============ STUDENT EXERCISE ============
# TODO: Identify the hash algorithm and find the matching message
#
# Steps:
#   1. Count the hex characters in the hash below
#   2. Consult the table at the top of Module 2 to identify the algorithm
#   3. Hash each candidate message with that algorithm
#   4. Find which one matches (case-sensitive!)
#
# Expected: SHA-1 (40 chars = 160 bits), and one specific capitalization matches
# ==========================================

target_hash = 'C4F3ACA26A244839355883DDC951083C1056FFBC'
candidates  = ['Stevens', 'stevens', 'STEVENS']

print(f'Hash length: {len(target_hash)} hex chars = {len(target_hash)*4} bits')
print('Algorithm hint: see the table in Module 2 header')
print()

# TODO: loop over candidates, hash each one with the identified algorithm,
#       compare to target_hash (case-insensitive comparison)
for candidate in candidates:
    # h = hashlib.???(candidate.encode()).hexdigest().upper()
    # print(f'{candidate}: {h}  match={h == target_hash}')
    pass


## Going further: RSA and homomorphic encryption

A closer look at how RSA is built, then a first taste of computing on encrypted data.

### Extended GCD and Modular Inverse


In [ ]:
def extended_gcd(a: int, b: int) -> tuple:
    """Extended Euclidean Algorithm.
    Returns (gcd, x, y) such that a*x + b*y = gcd(a, b).
    """
    if a == 0:
        return b, 0, 1
    gcd, x1, y1 = extended_gcd(b % a, a)
    return gcd, y1 - (b // a) * x1, x1

def mod_inverse(e: int, phi: int) -> int:
    """Compute d such that e*d ≡ 1 (mod phi)."""
    gcd, x, _ = extended_gcd(e % phi, phi)
    if gcd != 1:
        raise ValueError(f'Modular inverse does not exist: gcd({e},{phi})={gcd}')
    return x % phi

# Test
print(f'extended_gcd(35, 15) = {extended_gcd(35, 15)}')
print(f'mod_inverse(3, 352)  = {mod_inverse(3, 352)}')
# Verify: 3 * 235 mod 352 should = 1
print(f'Verify: 3 * 235 mod 352 = {3 * 235 % 352}')  # Expected: 1


### RSA Key Generation with Small Primes

We use tiny primes (p=17, q=23) for educational clarity.
Real RSA uses primes with 1024-4096 bits.


In [ ]:
import math

# Key generation
p   = 17
q   = 23
n   = p * q                          # modulus
phi = (p - 1) * (q - 1)             # Euler's totient
e   = 3                              # public exponent (must have gcd(e,phi)=1)
d   = mod_inverse(e, phi)            # private exponent

print('=== RSA Key Generation ===')
print(f'Primes      : p={p}, q={q}')
print(f'Modulus     : n = p*q = {n}')
print(f'Totient     : phi = (p-1)(q-1) = {phi}')
print(f'Public exp  : e = {e}  (gcd(e,phi) = {math.gcd(e,phi)})')
print(f'Private exp : d = {d}  (e*d mod phi = {(e*d) % phi})')
print(f'\nPublic key  : (e={e}, n={n})')
print(f'Private key : (d={d}, n={n})')


### RSA Encrypt and Decrypt


In [ ]:
def rsa_encrypt(m: int, e: int, n: int) -> int:
    """Encrypt integer m with public key (e, n)."""
    if m >= n:
        raise ValueError(f'Message {m} must be less than n={n}')
    return pow(m, e, n)  # Python's built-in modular exponentiation

def rsa_decrypt(c: int, d: int, n: int) -> int:
    """Decrypt ciphertext c with private key (d, n)."""
    return pow(c, d, n)

# Example: encrypt m=42
m = 42
c = rsa_encrypt(m, e, n)
m_recovered = rsa_decrypt(c, d, n)

print(f'Original message  : {m}')
print(f'Encrypted (c=m^e mod n): {c}')
print(f'Decrypted (m=c^d mod n): {m_recovered}')
print(f'Matches: {m == m_recovered}')  # Expected: True


### Encrypt a String Character by Character


In [ ]:
def rsa_encrypt_string(text: str, e: int, n: int) -> list:
    """Encrypt each character's ASCII value."""
    return [rsa_encrypt(ord(ch), e, n) for ch in text]

def rsa_decrypt_string(ciphertext: list, d: int, n: int) -> str:
    """Decrypt list of integers back to string."""
    return ''.join(chr(rsa_decrypt(c, d, n)) for c in ciphertext)

# Note: ASCII values must be < n=391
# All printable ASCII is < 128, so we are safe with n=391
msg = 'Hi!'
enc = rsa_encrypt_string(msg, e, n)
dec = rsa_decrypt_string(enc, d, n)
print(f'Original  : {msg}')
print(f'Encrypted : {enc}')
print(f'Decrypted : {dec}')


### RSA is NOT Additively Homomorphic

Some encryption schemes (like Paillier) support computation on ciphertexts.
RSA does NOT support addition, but it IS **multiplicatively homomorphic**.


In [ ]:
# RSA multiplicative homomorphism: E(m1) * E(m2) mod n = E(m1*m2)
m1, m2 = 3, 7
c1 = rsa_encrypt(m1, e, n)
c2 = rsa_encrypt(m2, e, n)

c_product   = (c1 * c2) % n
m_product   = rsa_decrypt(c_product, d, n)

print(f'E({m1}) * E({m2}) mod n = {c_product}')
print(f'Decrypted product      = {m_product}')
print(f'Expected ({m1}*{m2})         = {m1*m2}')
print(f'Multiplicatively homomorphic: {m_product == m1*m2}')

# BUT NOT additively homomorphic:
c_sum     = (c1 + c2) % n
m_sum     = rsa_decrypt(c_sum, d, n)
print(f'\nE({m1}) + E({m2}) mod n decrypts to: {m_sum}')
print(f'Expected ({m1}+{m2}): {m1+m2}')
print(f'Additively homomorphic: {m_sum == m1+m2}')  # Expected: False


### Fun Demo: Encrypt Pixel Values of a Tiny Image


In [ ]:
# Simulate a 3x3 grayscale 'image' as a numpy array
try:
    import numpy as np
    image = np.array([[10, 50, 200],
                      [30, 128, 75],
                      [99, 255, 0]], dtype=np.uint8)
    print('Original pixel values:')
    print(image)

    encrypted_image = np.array([[rsa_encrypt(int(px), e, n) for px in row]
                                 for row in image])
    print('\nEncrypted pixel values (integers mod n):')
    print(encrypted_image)

    decrypted_image = np.array([[rsa_decrypt(int(px), d, n) for px in row]
                                 for row in encrypted_image], dtype=np.uint8)
    print('\nDecrypted pixel values:')
    print(decrypted_image)
    print('Matches original:', np.array_equal(image, decrypted_image))
except ImportError:
    print('numpy not installed -- run: pip install numpy')
    print('Simulating without numpy:')
    flat = [10, 50, 200, 30, 128, 75, 99, 255, 0]
    enc = [rsa_encrypt(px, e, n) for px in flat]
    dec = [rsa_decrypt(c, d, n) for c in enc]
    print('Original :', flat)
    print('Decrypted:', dec)


### Student Challenge: RSA Mini-Calculation

**Given:** p=61, q=53, e=17

Show the full key generation and encrypt m=65.

Expected answers: n=3233, phi=3120, d=2753, c=2790


In [ ]:
# ============ STUDENT EXERCISE ============
# TODO: Complete the RSA calculation
#
# Given:
#   p = 61, q = 53, e = 17, m = 65
#
# Step 1: Compute n = p * q
# Step 2: Compute phi = (p-1) * (q-1)
# Step 3: Verify gcd(e, phi) == 1
# Step 4: Compute d = mod_inverse(e, phi)
# Step 5: Encrypt m: c = m^e mod n
# Step 6: Verify: decrypt c and check you get m back
#
# Expected: n=3233, phi=3120, d=2753, c=2790
# ==========================================

p_ch, q_ch, e_ch, m_ch = 61, 53, 17, 65

# TODO: fill in each step
# n_ch   = ?
# phi_ch = ?
# d_ch   = ?
# c_ch   = ?
# print(f'n={n_ch}, phi={phi_ch}, d={d_ch}, c={c_ch}')
# print(f'Decrypted: {rsa_decrypt(c_ch, d_ch, n_ch)}')


### Bonus: Computing on Encrypted Data (toy homomorphic encryption)

Fully Homomorphic Encryption (FHE) lets a computer do math on encrypted numbers without ever decrypting them. A hospital or a power company can send encrypted data to the cloud, the cloud computes on it, and only the data owner can read the result. Real systems use schemes such as Paillier or CKKS (for example the TenSEAL library).

You saw above that RSA is *multiplicatively* homomorphic. The toy below shows the *additive* idea: adding two ciphertexts yields an encryption of the sum. It is an illustration only and is **not** secure.

In [ ]:
# Toy ADDITIVE homomorphic encryption -- ILLUSTRATION ONLY (not secure).
# Real privacy-preserving ML uses Paillier or CKKS (e.g., the TenSEAL library).
import secrets

MOD = 1_000_003                      # public prime modulus
secret_key = secrets.randbelow(MOD)  # private key, kept by the data owner

def encrypt(m, key=secret_key):
    return (m + key) % MOD

def decrypt(c, n_terms=1, key=secret_key):
    return (c - n_terms * key) % MOD

a, b = 7, 5
ca, cb = encrypt(a), encrypt(b)
print(f'plaintext  a={a}, b={b}')
print(f'ciphertext a={ca}, b={cb}  (look random)')

c_sum = (ca + cb) % MOD              # the cloud adds ciphertexts, knowing neither a nor b
result = decrypt(c_sum, n_terms=2)  # two ciphertexts added -> subtract the key twice
print('decrypt(ca + cb) =', result, ' and  a + b =', a + b)
assert result == a + b
print('The cloud computed a+b on encrypted data and never saw a or b.')
